# SimMIM Pre-training for SwinUnet Wildfire Model

Self-supervised pre-training using channel-group masking on fire-agnostic satellite data.
The model learns to reconstruct masked channel groups from visible ones, learning
cross-feature spatial relationships (e.g., predict weather from terrain).

**Prerequisites:**
- Pre-training TIFs in GCS bucket `lmudl-wildfire-compilation-bucket/WildfireSpreadTS/pretrain/`
- **Runtime -> Change runtime type -> GPU** (T4 or better)

**After pre-training:** load the saved encoder+decoder weights into SwinUnet and fine-tune on the fire dataset.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

In [ ]:
REPO_ORG   = "amindell11"
REPO_NAME  = "wildfire-ts-swin"
REPO_BRANCH = "feature/simim-pretrain"

GCS_BUCKET  = "lmudl-wildfire-compilation-bucket"
GCS_PREFIX  = "WildfireSpreadTS/pretrain"
LOCAL_TIF_DIR = "/content/pretrain_tifs"

OUTPUT_DIR  = "/content/drive/MyDrive/wildfire_runs/pretrain"

MAX_EPOCHS        = 200
BATCH_SIZE        = 32
BASE_LR           = 1.5e-4
WEIGHT_DECAY      = 0.05
WARMUP_EPOCHS     = 10
MASK_PROB         = 0.2    # ~5 features masked per sample (out of 24)
CROP_SIDE_LENGTH  = 128
CROPS_PER_TIF     = 4      # random crops per TIF per epoch (multiplies effective dataset size)
STATS_YEARS       = (2020, 2021)
NUM_WORKERS       = 4
SEED              = 42

CHECKPOINT_INTERVAL = 20

## 3. Download TIFs from GCS to local disk

Copies all pre-training TIFs to Colab's local NVMe for fast I/O during training.

In [ ]:
!pip install -q gcsfs
from google.colab import auth
auth.authenticate_user()

import os
os.makedirs(LOCAL_TIF_DIR, exist_ok=True)

!gsutil -m cp -r gs://{GCS_BUCKET}/{GCS_PREFIX}/* {LOCAL_TIF_DIR}/

# Count downloaded TIFs
import glob
tifs = glob.glob(f"{LOCAL_TIF_DIR}/**/*.tif", recursive=True)
print(f"Downloaded {len(tifs)} TIF files")

## 4. Clone repo and install dependencies

In [ ]:
_repo_url = f"https://github.com/{REPO_ORG}/{REPO_NAME}.git"
!rm -rf /content/{REPO_NAME}
!git clone -b {REPO_BRANCH} {_repo_url} /content/{REPO_NAME}
!pip install -q -r /content/{REPO_NAME}/requirements.txt
!pip install -q rasterio
!git -C /content/{REPO_NAME} log --oneline -5

In [ ]:
import sys
sys.path.insert(0, f'/content/{REPO_NAME}')

## 5. Detect accelerator

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print("WARNING: No GPU detected.")

## 6. Build Model & Preview Masking

In [ ]:
import os
import random
import types
import numpy as np
import torch
import torch.backends.cudnn as cudnn

from config import get_config
from networks.mae_swin_unet import SimMIMSwinUnet
from trainer_mae_pretrain import trainer_mae_pretrain
from datasets.wildfire.pretrain_dataset import PretrainDataset

if torch.cuda.is_available():
    cudnn.benchmark = True
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Build config (reuse existing YAML for architecture params)
config_args = types.SimpleNamespace(
    cfg=f'/content/{REPO_NAME}/configs/swin_tiny_patch4_window4_128_wildfire.yaml',
    opts=['MODEL.SWIN.IN_CHANS', '40', 'MODEL.SWIN.N_TIMESTEPS', '1', 'MODEL.PRETRAIN_CKPT', 'None'],
    batch_size=None, zip=False, cache_mode='part', resume=None,
    accumulation_steps=None, use_checkpoint=False, amp_opt_level='',
    tag=None, eval=False, throughput=False,
)
config = get_config(config_args)

# Build model
model = SimMIMSwinUnet(
    config, img_size=CROP_SIDE_LENGTH, mask_prob=MASK_PROB,
    use_factored_embed=False,
).to(DEVICE)

# Build dataset
dataset = PretrainDataset(
    tif_dir=LOCAL_TIF_DIR,
    crop_side_length=CROP_SIDE_LENGTH,
    stats_years=STATS_YEARS,
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Mask prob: {MASK_PROB} (~{int(24 * MASK_PROB)} features masked per sample)")
print(f"Pre-training samples: {len(dataset)}")

In [ ]:
import matplotlib.pyplot as plt
from networks.mae_swin_unet import MASK_UNITS, MASK_UNIT_NAMES

# Preview: run one forward pass with random weights to show masking structure.
# Reconstruction will be garbage (untrained) — just validates the pipeline.

# Show 8 representative features across different types
VIS_FEATURES = {
    'NDVI':       3,
    'Precip':     5,
    'Temp Max':   9,
    'Elevation': 14,
    'PDSI':      15,
    'Landcover': 16,   # first of 17 one-hot channels
    'Fcst Temp': 36,
    'Wind Spd':   6,
}

# Load one real sample
sample = dataset[0].unsqueeze(0).to(DEVICE)  # (1, 40, 128, 128)

model.eval()
with torch.no_grad():
    loss, pred, mask = model(sample)

x_orig = sample[0].cpu()
x_masked = (sample * (1.0 - mask))[0].cpu()
x_pred = pred[0].cpu()
mask_0 = mask[0].cpu()

def normalize_for_display(t):
    lo, hi = t.min(), t.max()
    if hi - lo < 1e-8:
        return torch.zeros_like(t)
    return (t - lo) / (hi - lo)

n_ch = len(VIS_FEATURES)
fig, axes = plt.subplots(3, n_ch, figsize=(2.8 * n_ch, 8.5))

row_labels = ['Original', 'Masked Input', 'Reconstruction\n(untrained)']
for col, (name, ch) in enumerate(VIS_FEATURES.items()):
    is_masked = mask_0[ch, 0, 0].item()
    status = "MASKED" if is_masked else "visible"

    imgs = [
        normalize_for_display(x_orig[ch]),
        normalize_for_display(x_masked[ch]),
        normalize_for_display(x_pred[ch]),
    ]

    for row in range(3):
        ax = axes[row, col]
        ax.imshow(imgs[row].numpy(), cmap='viridis', vmin=0, vmax=1)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            color = 'red' if is_masked else 'green'
            ax.set_title(f'{name}\n({status})', fontsize=9, color=color)
        if col == 0:
            ax.set_ylabel(row_labels[row], fontsize=10)

plt.suptitle(f'Per-Feature Masking Preview (untrained, mask_prob={MASK_PROB})', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Print full mask summary
print(f"\nLoss (untrained): {loss.item():.4f}")
n_masked = sum(1 for u, chs in enumerate(MASK_UNITS) if mask_0[chs[0], 0, 0] == 1.0)
print(f"Features masked: {n_masked}/{len(MASK_UNITS)}")
for u, (chs, name) in enumerate(zip(MASK_UNITS, MASK_UNIT_NAMES)):
    status = 'MASKED' if mask_0[chs[0], 0, 0] == 1.0 else 'visible'
    print(f"  {name:20s}: {status}")

## 7. Run Pre-training

Automatically resumes from the latest checkpoint if one exists in `OUTPUT_DIR`.  
Set `NO_RESUME = True` to train from scratch.

In [ ]:
NO_RESUME = False  # set True to ignore existing checkpoints and train from scratch

%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/pretrain_log

args = types.SimpleNamespace(
    tif_dir=LOCAL_TIF_DIR,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    base_lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),
    warmup_epochs=WARMUP_EPOCHS,
    num_workers=NUM_WORKERS,
    crop_side_length=CROP_SIDE_LENGTH,
    stats_years=STATS_YEARS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    no_resume=NO_RESUME,
    crops_per_tif=CROPS_PER_TIF,
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer_mae_pretrain(args, model, OUTPUT_DIR, device=DEVICE)

## 8. Next steps

Pre-trained weights are saved to Google Drive:
- `{OUTPUT_DIR}/pretrain_best.pth` — best val loss (encoder+decoder only)
- `{OUTPUT_DIR}/pretrain_encoder_decoder.pth` — final epoch (encoder+decoder only)
- `{OUTPUT_DIR}/pretrain_ckpt_epoch*.pth` — periodic full checkpoints

To fine-tune on the fire dataset, open the `train_and_test_colab.ipynb` notebook and set `PRETRAIN_WEIGHTS` to the path above.